# 11 - Customer Report

This notebook creates a reusable customer-level report view.

The report combines customer attributes with sales behavior and calculates:
- customer age and age group
- customer segment
- total orders
- total sales
- total quantity purchased
- total products purchased
- recency
- average order value
- average monthly spend

In [0]:
%sql
/*
Customer Report View

Purpose:
    Create a reusable customer-level analytical view by combining customer
    attributes with sales behavior.

Output:
    One row per customer with demographic attributes, sales metrics,
    lifecycle metrics, and customer segmentation.
*/

CREATE OR REPLACE VIEW datawarehouseanalytics_gold.report_customers AS

WITH base_query AS (
    SELECT
        f.order_number,
        f.product_key,
        f.order_date,
        f.sales_amount,
        f.quantity,
        c.customer_key,
        c.customer_number,
        CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
        FLOOR(MONTHS_BETWEEN(CURRENT_DATE(), c.birthdate) / 12) AS age
    FROM datawarehouseanalytics_gold.fact_sales f
    LEFT JOIN datawarehouseanalytics_gold.dim_customers c
        ON f.customer_key = c.customer_key
    WHERE f.order_date IS NOT NULL
),

customer_aggregation AS (
    SELECT
        customer_key,
        customer_number,
        customer_name,
        age,
        COUNT(DISTINCT order_number) AS total_orders,
        SUM(sales_amount) AS total_sales,
        SUM(quantity) AS total_quantity,
        COUNT(DISTINCT product_key) AS total_products,
        MIN(order_date) AS first_order_date,
        MAX(order_date) AS last_order_date,
        ROUND(MONTHS_BETWEEN(MAX(order_date), MIN(order_date)), 0) AS lifespan_months
    FROM base_query
    GROUP BY
        customer_key,
        customer_number,
        customer_name,
        age
)

SELECT
    customer_key,
    customer_number,
    customer_name,
    age,

    CASE 
        WHEN age < 20 THEN 'Under 20'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        ELSE '50 and above'
    END AS age_group,

    CASE 
        WHEN lifespan_months >= 12 AND total_sales > 5000 THEN 'VIP'
        WHEN lifespan_months >= 12 AND total_sales <= 5000 THEN 'Regular'
        ELSE 'New'
    END AS customer_segment,

    first_order_date,
    last_order_date,

    ROUND(MONTHS_BETWEEN(CURRENT_DATE(), last_order_date), 0) AS recency_months,

    total_orders,
    total_sales,
    total_quantity,
    total_products,
    lifespan_months,

    CASE 
        WHEN total_orders = 0 THEN 0
        ELSE ROUND(total_sales / total_orders, 2)
    END AS avg_order_value,

    CASE 
        WHEN lifespan_months = 0 THEN total_sales
        ELSE ROUND(total_sales / lifespan_months, 2)
    END AS avg_monthly_spend

FROM customer_aggregation;

In [0]:
%sql
SELECT *
FROM datawarehouseanalytics_gold.report_customers
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS total_customers_in_report
FROM datawarehouseanalytics_gold.report_customers;

In [0]:
%sql
SELECT
    customer_segment,
    COUNT(*) AS total_customers,
    SUM(total_sales) AS total_sales,
    ROUND(AVG(avg_order_value), 2) AS avg_order_value,
    ROUND(AVG(avg_monthly_spend), 2) AS avg_monthly_spend
FROM datawarehouseanalytics_gold.report_customers
GROUP BY customer_segment
ORDER BY total_sales DESC;